Some common file object methods

1. ```read(size)``` will read size characters/bytes as a string
2. ```write(string)``` will write string/bytes to a file
3. ```readline()``` will read a string until and including the next newline character is met
4. ```readlines()``` will return a list of all lines of a file
5. ```writelines()``` will write a list of lines to a file
6. ```flush()``` will try to make sure that the changes made to a file are written to disk immediately

In [1]:
f=open('30-Basic_file_processing.ipynb', 'r')  # Let's open this notebook file,
                                              # which is essentially a text file.
                                               # So you can open it in a texteditor as well.
for i in range(5):
    line=f.readline()
    print(f'Line {i}: {line}', end=' ')
f.close()

Line 0: {
 Line 1:  "cells": [
 Line 2:   {
 Line 3:    "metadata": {},
 Line 4:    "cell_type": "markdown",
 

It is easy to forget to close the file. One can use a ```context manager``` to solve this problem

A context manager is created with the ```with``` statement

After the indented block of the ```with``` statement exits, the file will be automatically closed.

In [3]:
with open('30-Basic_file_processing.ipynb', 'r') as f: # the file will be automatically                                                        #closed,when the with block exits
    for i in range(5):
        line=f.readline()
        print(f'Line {i}: {line}', end=' ')

Line 0: {
 Line 1:  "cells": [
 Line 2:   {
 Line 3:    "metadata": {},
 Line 4:    "cell_type": "markdown",
 

The file object is iterable. This means that we can iterate through the lines in the file using a for loop, like in the below example:

In [6]:
max_length=0

with open('30-Basic_file_processing.ipynb', 'r') as f:
    for line in f:
        if len(line)>max_length:
            max_length=len(line)

print(f'The longest line in tis file has length {max_length}')

The longest line in tis file has length 453


## File Methods — Simply, With Examples

Six methods, and you've already used a few! Let's go through each with a shared example file.

```
sample.txt:
line one
line two
line three
```

---

### 1. `read(size)` — Read a Fixed Number of Characters

Without an argument, `read()` grabs the **whole file** (you've used this before). With `size`, it only pulls that many characters:

```python
with open("sample.txt") as f:
    print(f.read(4))    # → 'line'      first 4 characters only
    print(f.read(4))    # → ' one'      NEXT 4 characters (cursor moved forward!)
```

**Key idea:** the file keeps an internal "cursor" — each `read()` call continues from wherever the previous one stopped, like a bookmark moving through the text.

```python
with open("sample.txt") as f:
    print(f.read())      # → entire file (default: no size limit)
```

---

### 2. `write(string)` — Write Text to a File

```python
with open("output.txt", "w") as f:
    f.write("Hello\n")
    f.write("World\n")
```

`output.txt` now contains:
```
Hello
World
```

Notice — `write()` does **not** add a newline automatically. If you don't include `\n` yourself, the text just runs together on one line:

```python
f.write("Hello")
f.write("World")
# → "HelloWorld"    ← no separation!
```

---

### 3. `readline()` — Read ONE Line at a Time

Reads up to and **including** the next `\n`:

```python
with open("sample.txt") as f:
    print(f.readline())    # → 'line one\n'
    print(f.readline())    # → 'line two\n'    (cursor moved to the next line)
    print(f.readline())    # → 'line three\n'  (or 'line three' if no trailing \n)
    print(f.readline())    # → ''              empty string = end of file
```

Each call gives you **one more line**, moving the cursor forward — same bookmark idea as `read(size)`, just measured in lines instead of characters.

**Empty string `''` means "nothing left to read"** — useful for a manual loop:

```python
with open("sample.txt") as f:
    line = f.readline()
    while line:
        print(line, end="")
        line = f.readline()
```

(Though in practice, `for line in f:` — which you've used throughout this conversation — does this automatically and is preferred.)

---

### 4. `readlines()` — Get ALL Lines as a List

```python
with open("sample.txt") as f:
    lines = f.readlines()
    print(lines)
    # → ['line one\n', 'line two\n', 'line three\n']
```

You've seen this too — it loads the **entire file at once** into a list, one string per line, each keeping its trailing `\n`. Compare with `for line in f:` (lazy, one line at a time) — `readlines()` is the eager, all-at-once version, same distinction as `range()` vs a real list.

---

### 5. `writelines(list)` — Write a List of Lines at Once

The mirror image of `readlines()`:

```python
lines = ["apple\n", "banana\n", "cherry\n"]

with open("fruits.txt", "w") as f:
    f.writelines(lines)
```

`fruits.txt`:
```
apple
banana
cherry
```

**Important gotcha:** despite the name, `writelines()` does **NOT** add newlines for you — just like `write()`. If your list items don't already have `\n` at the end, everything gets mashed together:

```python
f.writelines(["apple", "banana", "cherry"])
# → "applebananacherry"    ← no newlines added!
```

You must include `\n` yourself in each string, or join first:

```python
f.writelines(f"{fruit}\n" for fruit in ["apple", "banana", "cherry"])
```

---

### 6. `flush()` — Force Immediate Save to Disk

Remember this concept from way back — the **stream and buffer** discussion with `print()`? Same idea here, applied to files.

When you `write()` to a file, Python often **doesn't immediately** save it to the physical disk — it holds the data in a temporary memory buffer for efficiency, and writes in batches.

```python
with open("output.txt", "w") as f:
    f.write("important data")
    f.flush()          # force it onto disk RIGHT NOW, don't wait
```

**Why does this matter?** If your program crashes (or the power goes out) before the buffer naturally empties, unflushed data can be **lost** — even though you already called `write()`. `flush()` is an insurance policy: *"don't wait, save this immediately."*

In practice, you rarely need to call this manually — `with open(...) as f:` automatically flushes and closes the file when the block ends. `flush()` matters mainly for long-running programs (like a live log file) where you want data saved **before** the file eventually closes.

---

### Summary Table

| Method | Reads/Writes | Returns/Effect |
|---|---|---|
| `read(size)` | reads | a string of `size` characters (or whole file if omitted) |
| `write(s)` | writes | writes `s` — no automatic `\n` |
| `readline()` | reads | ONE line (with its `\n`), `''` when file is exhausted |
| `readlines()` | reads | ALL lines, as a list |
| `writelines(list)` | writes | writes each item — no automatic `\n` between them |
| `flush()` | — | forces buffered writes onto disk immediately |

---

### The One-Sentence Summary

> Reading methods (`read`, `readline`, `readlines`) move an internal cursor through the file, returning either a fixed chunk, one line, or everything as a list. Writing methods (`write`, `writelines`) never add newlines automatically — you supply your own `\n`. `flush()` is the manual override for the buffering behavior you first met with `print(flush=True)` — pushing data to disk before the normal automatic flush would happen. 🎯